# Exploratory Analysis suite

Version 1
- word count
- n-grams
- tf-idf (log?)
- collocation explorer

Version 2
- topic clusters
- semantic similarity
- embeddings?

In [2]:
import pandas as pd # used to work with dataframes, which is important to store the data in a structured way and to manipulate it easily
from nltk.corpus import stopwords # used to remove stopwords
import sqlite3 # used to work with the database

import matplotlib.pyplot as plt # used to visualize the data, which is important to understand it and to communicate the results

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer


# import gensim # used for topic modeling, which is important to extract the main topics from the laws and to understand their content
# import spacy # used for named entity recognition, which is important to extract the entities from the laws and to understand their content

In [4]:
LOCATION_SPEECHES = ("../../data/processed/matched_speeches.csv")
LOCATION_DB = ("../../data/chilean_polispeech.db")

# # LOCATION_SPEECHES = ("C:\Users\jorge\Documents\Mis documentos\Uni Konstanz\1. ICSS + Python\Project\chilean-polispeech-nlp\data\processed\matched_speeches.csv")
# LOCATION_SPEECHES = ("C:\\Users\\jorge\\Documents\\Mis documentos\\Uni Konstanz\\1. ICSS + Python\\Project\\chilean-polispeech-nlp\\data\\processed\\matched_speeches.csv")
# LOCATION_DB = ("C:\\Users\\jorge\\Documents\\Mis documentos\\Uni Konstanz\\1. ICSS + Python\\Project\\chilean-polispeech-nlp\\data\\chilean_polispeech.db")

In [5]:
matched_speeches = pd.read_csv(LOCATION_SPEECHES)

In [6]:
# #Idea on how to get the data for the analysis and the dashboard.
# # get_corpus_slice(
#     law_numbers=None,
#     dipids=None,
#     sex_values=None,
#     regions=None,
#     parties=None,
#     substantive_only=True
# ) 

# top_ngrams(df, text_col="speech_clean", n=2, top_k=20)
# tfidf_keywords(df, text_col="speech_clean", top_k=20)
# corpus_stats(df, text_col="speech_clean")

### Start of analysis

top_k = 10 means 10 results

min_df=2 means that terms have to appear in at least 2 documents (how rare words are)

ngram_range= (1,1) is words, (2,2) is bigrams, (1,2) words and bigrams

In [6]:
con = sqlite3.connect(LOCATION_DB)

analysis_df = pd.read_sql_query("""
    SELECT
        s.speech_id,
        s.speech_raw,
        s.speech_clean,
        s.dipid,
        ls.law_title,
        ls.law_number,
        ls.section_key,
        ls.discussion_date,
        d.name,
        d.lastname_1,
        d.lastname_2,
        d.sex_value,
        m.party_id,
        p.party_name,
        p.party_alias
    FROM speeches AS s
    JOIN law_sections AS ls
        ON s.law_section_id = ls.law_section_id
    LEFT JOIN deputies AS d
        ON s.dipid = d.dipid
    LEFT JOIN militancies AS m
        ON s.dipid = m.dipid
       AND date(ls.discussion_date) >= date(m.date_start)
       AND (
            m.date_end IS NULL
            OR date(ls.discussion_date) <= date(m.date_end)
       )
    LEFT JOIN parties AS p
        ON m.party_id = p.party_id
""", con)

analysis_df.head()

,speech_id,speech_raw,speech_clean,dipid,law_title,law_number,section_key,discussion_date,name,lastname_1,lastname_2,sex_value,party_id,party_name,party_alias
0,5808,"Señor Presidente, para nosotros es demasiado i...","presidente, demasiado importante ir ayuda dire...",1015.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Jorge,Brito,Hasbún,1,RD,Revolución Democrática,RD
1,5809,"Señor Presidente, me gustaría expresar lo defi...","presidente, gustaría expresar deficitario proy...",1126.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Felipe,Donoso,Castro,1,UDI,Unión Demócrata Independiente,UDI
2,5810,"Señor Presidente, junto con saludar al ministr...","presidente, junto saludar energía, quiero deci...",1161.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Francisco,Pulgar,Castillo,1,IND,Independientes,IND
3,5811,"Señor Presidente, este es un proyecto malo, as...","presidente, malo, así simple: malo. aplaudir f...",1019.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,José Miguel,Castro,Bascuñán,1,RN,Renovación Nacional,RN
4,5812,"Señor Presidente, aprobaremos este proyecto de...","presidente, aprobaremos ley, dentro contexto c...",872.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Jaime,Mulet,Martínez,1,FRVS,Federación Regionalista Verde Social,FRVS


In [9]:
analysis_df

,speech_id,speech_raw,speech_clean,dipid,law_title,law_number,section_key,discussion_date,name,lastname_1,lastname_2,sex_value,party_id,party_name,party_alias
0,4795,"Señor Presidente, para nosotros es demasiado i...","presidente, demasiado importante ir ayuda dire...",1015.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Jorge,Brito,Hasbún,1,RD,Revolución Democrática,RD
1,4796,"Señor Presidente, me gustaría expresar lo defi...","presidente, gustaría expresar deficitario proy...",1126.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Felipe,Donoso,Castro,1,UDI,Unión Demócrata Independiente,UDI
2,4797,"Señor Presidente, junto con saludar al ministr...","presidente, junto saludar energía, quiero deci...",1161.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Francisco,Pulgar,Castillo,1,IND,Independientes,IND
3,4798,"Señor Presidente, este es un proyecto malo, as...","presidente, proyecto malo, así simple: malo. a...",1019.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,José Miguel,Castro,Bascuñán,1,RN,Renovación Nacional,RN
4,4799,"Señor Presidente, aprobaremos este proyecto de...","presidente, aprobaremos proyecto ley, dentro c...",872.0,Crea un Fondo de Estabilización de Tarifas y e...,21.472,1-5,2022-06-07,Jaime,Mulet,Martínez,1,FRVS,Federación Regionalista Verde Social,FRVS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1011,5803,"Señor Presidente, artículo 77. Estamos sesiona...","presidente, artículo sesionando menos cuarenta...",917.0,Regula la extracción de áridos y modifica los ...,21.8,2-5,2024-12-17,Gastón,Von Mühlenbrock,Zamora,1,UDI,Unión Demócrata Independiente,UDI
1012,5804,"Señor Presidente, qué importante es contar con...","presidente, importante contar regulación activ...",1111.0,Regula la extracción de áridos y modifica los ...,21.8,2-5,2024-12-17,Fernando,Bórquez,Montecinos,1,UDI,Unión Demócrata Independiente,UDI
1013,5805,"Señor Presidente, respecto del proyecto, acá y...","presidente, respecto proyecto, acá señalado pu...",1134.0,Regula la extracción de áridos y modifica los ...,21.8,2-5,2024-12-17,Andrés,Jouannet,Valderrama,1,AMA,Movimiento Amarillos por Chile,AMA
1014,5806,"Señor Presidente, por su intermedio saludo al ...","presidente, intermedio saludo representante ej...",1075.0,Regula la extracción de áridos y modifica los ...,21.8,2-5,2024-12-17,Diego,Schalper,Sepúlveda,1,RN,Renovación Nacional,RN


In [7]:
analysis_df = pd.read_sql_query("""
SELECT
    s.speech_id,
    s.speech_raw,
    s.speech_clean,
    s.dipid,
    ls.law_number,
    ls.section_key,
    ls.discussion_date,
    d.name,
    d.lastname_1,
    d.lastname_2,
    d.sex_value,
    dist.region,
    dist.comunas
FROM speeches s
JOIN law_sections ls
    ON s.law_section_id = ls.law_section_id
LEFT JOIN deputies d
    ON s.dipid = d.dipid
LEFT JOIN districts dist
    ON s.dipid = dist.dipid
WHERE s.speech_clean IS NOT NULL
  AND TRIM(s.speech_clean) <> ''
""", con)
analysis_df.head()

,speech_id,speech_raw,speech_clean,dipid,law_number,section_key,discussion_date,name,lastname_1,lastname_2,sex_value,region,comunas
0,5808,"Señor Presidente, para nosotros es demasiado i...","presidente, demasiado importante ir ayuda dire...",1015.0,21.472,1-5,2022-06-07,Jorge,Brito,Hasbún,1,Región De Valparaíso,"Algarrobo, Cartagena, Casablanca, Concón, El Q..."
1,5809,"Señor Presidente, me gustaría expresar lo defi...","presidente, gustaría expresar deficitario proy...",1126.0,21.472,1-5,2022-06-07,Felipe,Donoso,Castro,1,Región Del Maule,"Constitución, Curepto, Curicó, Empedrado, Hual..."
2,5810,"Señor Presidente, junto con saludar al ministr...","presidente, junto saludar energía, quiero deci...",1161.0,21.472,1-5,2022-06-07,Francisco,Pulgar,Castillo,1,Región Del Maule,"Constitución, Curepto, Curicó, Empedrado, Hual..."
3,5811,"Señor Presidente, este es un proyecto malo, as...","presidente, malo, así simple: malo. aplaudir f...",1019.0,21.472,1-5,2022-06-07,José Miguel,Castro,Bascuñán,1,Región De Antofagasta,"Antofagasta, Calama, María Elena, Mejillones, ..."
4,5812,"Señor Presidente, aprobaremos este proyecto de...","presidente, aprobaremos ley, dentro contexto c...",872.0,21.472,1-5,2022-06-07,Jaime,Mulet,Martínez,1,Región De Atacama,"Alto Del Carmen, Caldera, Chañaral, Copiapó, D..."


Filter function

In [ ]:
# filter function

# In the dashboard we want to be able to select which speeches to play with
def get_corpus_slice(df, laws=None, dipids=None, sexes=None, regions=None):
    # start from a copy so the original dataframe stays unchanged
    out = df.copy()

    # keep only selected laws if provided
    if laws:
        out = out[out["law_number"].isin(laws)]

    # keep only selected deputies if provided
    if dipids:
        out = out[out["dipid"].isin(dipids)]

    # keep only selected sex values if provided
    if sexes:
        out = out[out["sex_value"].isin(sexes)]

    # keep only selected regions if the column exists and filter is provided
    if regions and "region" in out.columns:
        out = out[out["region"].isin(regions)]

    # remove empty speeches
    out = out[out["speech_clean"].notna()]
    out = out[out["speech_clean"].str.strip() != ""]

    return out

#### Corpus stats

In [ ]:
# it would be good to have a way to know how big the corpus is

def corpus_stats(df):
    # count words in each speech
    word_counts = df["speech_clean"].str.split().str.len()

    # return a small summary
    return pd.Series({
        "n_speeches": len(df),
        "n_unique_speakers": df["dipid"].nunique(),
        "total_words": word_counts.sum(),
        "avg_words_per_speech": round(word_counts.mean(), 2)
    })

In [13]:
corpus_stats(analysis_df)

n_speeches                1013.00
n_unique_speakers          142.00
total_words             163660.00
avg_words_per_speech       161.56
dtype: float64

#### Top n-grams (top words)

This function allow to check the words that appear the most (or n-grams)

In [14]:
def top_ngrams(df, text_col="speech_clean", top_k=20, min_df=2, ngram_range=(1, 1)): # also top words
    """
    Get top words or n-grams by raw count.
    """
    vec = CountVectorizer(
        min_df=min_df,
        ngram_range=ngram_range
    )

    X = vec.fit_transform(df[text_col])

    terms = vec.get_feature_names_out()
    counts = X.sum(axis=0).A1

    result = pd.DataFrame({
        "term": terms,
        "count": counts
    })

    return result.sort_values("count", ascending=False).head(top_k)

In [15]:
# top words
top_ngrams(analysis_df, text_col="speech_clean", ngram_range=(1, 1), min_df=2)

# top bigrams
top_ngrams(analysis_df, text_col="speech_clean", ngram_range=(2, 2), min_df=3)

# top trigrams from raw text
top_ngrams(analysis_df, text_col="speech_raw", ngram_range=(3, 3), min_df=2)

,term,count
37949,proyecto de ley,579
18790,este proyecto de,386
25551,la región de,201
36255,por lo tanto,196
36351,por su intermedio,186
9902,de nuestro país,180
24273,la comisión de,175
8617,de este proyecto,175
15996,en nuestro país,156
15673,en la comisión,144


#### TF-IDF 
This compares which words are more characteristic of group A than group B.
What's distinctive or characteristic (concentrated)?
Not "who says it more in total"
Terms can be not very frequent overall but rank high

In [16]:

def compare_tfidf_groups(df, group_col, group_a, group_b, text_col="speech_clean", top_k=20, min_df=2, ngram_range=(1, 1)):
    """
    Compare average TF-IDF scores between two groups.
    """
    # build the TF-IDF representation of the texts
    vec = TfidfVectorizer( # split into terms, a term is a word or n-gram depending on the ngram_range
        min_df=min_df, 
        ngram_range=ngram_range # this decides if it is n-gram or just a word
    )

    # fit the vectorizer and transform the texts into a matrix
    X = vec.fit_transform(df[text_col])
    
    # get the actual words (or n-grams) used
    words = vec.get_feature_names_out()

    # turn the matrix into a normal dataframe
    tfidf_df = pd.DataFrame(X.toarray(), columns=words, index=df.index)

    # add the group label to each row
    tfidf_df[group_col] = df[group_col].values
    
    # average the TF-IDF score of each term inside each group
    grouped = tfidf_df.groupby(group_col).mean()

    # build a result table
    result = pd.DataFrame({
        "term": grouped.columns,
        f"{group_a}_tfidf": grouped.loc[group_a].values,
        f"{group_b}_tfidf": grouped.loc[group_b].values
    })

    # positive difference = more characteristic of group_a
    # negative difference = more characteristic of group_b
    result["difference"] = result[f"{group_a}_tfidf"] - result[f"{group_b}_tfidf"]

    # absolute difference helps sort the strongest contrasts
    result["abs_difference"] = result["difference"].abs()

    # here we select the top_k terms with the biggest difference in usage between the two groups
    return result.sort_values("abs_difference", ascending=False).head(top_k)

In [17]:
compare_tfidf_groups(analysis_df, group_col= "sex_value", group_a="0", group_b="1", text_col="speech_clean", top_k=20, min_df=2, ngram_range=(2, 2))
# how to read the results: the first column is the term (in this case, bigrams), the second and third columns are the average TF-IDF score of that term for each group (in this case, sex_value 0 and 1, which correspond to the two groups)
# the fourth column is the difference in TF-IDF score between the two groups (positive means group_a uses it more, negative means group_b uses it more), and the last column is the absolute difference, which is used to sort the terms by how strongly they are associated with one group or the other.

,term,0_tfidf,1_tfidf,difference,abs_difference
20921,punto reglamento,0.011270,0.000000,0.011270,0.011270
20425,proyecto ley,0.015534,0.024690,-0.009156,0.009156
21964,reglamento palabra,0.008928,0.000000,0.008928,0.008928
16583,niños niñas,0.012492,0.003861,0.008631,0.008631
18908,policía investigaciones,0.003260,0.010522,-0.007263,0.007263
18531,pescadores artesanales,0.004833,0.011903,-0.007070,0.007070
17367,palabra daniel,0.006774,0.000000,0.006774,0.006774
17383,palabra miguel,0.006766,0.000000,0.006766,0.006766
4143,clase media,0.000969,0.007587,-0.006618,0.006618
15410,miguel ángel,0.006439,0.000000,0.006439,0.006439


Difference is simple 0_tfidf - 1 1_tfidf

#### Compare two groups
Which words/n-grams appear more often in Group a vs group B

This is pure frequency, but can be biases because bigger groups (more members) use more words

In [18]:
def compare_groups_totalwords(df, group_col, group_a, group_b, text_col="speech_clean", top_k=20, min_df=2, ngram_range=(1, 1)):
    # keep only the two groups we want to compare
    sub = df[df[group_col].isin([group_a, group_b])].copy()

    # build term counts
    vec = CountVectorizer(
        ngram_range=ngram_range,
        min_df=min_df
    )

    X = vec.fit_transform(sub[text_col])
    
    terms = vec.get_feature_names_out()

    # convert matrix to dataframe
    term_df = pd.DataFrame(X.toarray(), columns=terms, index=sub.index)

    # add the group label back
    term_df[group_col] = sub[group_col].values

    # sum counts inside each group
    grouped = term_df.groupby(group_col).sum()

    # create a comparison table
    result = pd.DataFrame({
        "term": grouped.columns,
        f"{group_a}_count": grouped.loc[group_a].values,
        f"{group_b}_count": grouped.loc[group_b].values
    })

    # difference tells us which group uses the term more
    result["difference"] = result[f"{group_a}_count"] - result[f"{group_b}_count"]
    result["abs_difference"] = result["difference"].abs()

    return result.sort_values("abs_difference", ascending=False).head(top_k)

In [19]:
# which terms appear more often in one group more than the other?
compare_groups_totalwords(analysis_df, group_col="sex_value", group_a="0", group_b="1", ngram_range=(2, 2), top_k=15)

,term,0_count,1_count,difference,abs_difference
20389,proyecto ley,151,432,-281,281
18873,policía investigaciones,18,96,-78,78
4134,clase media,8,86,-78,78
16552,niños niñas,120,48,72,72
10031,favor proyecto,33,104,-71,71
18497,pescadores artesanales,26,89,-63,63
18476,pesca artesanal,43,105,-62,62
5570,crimen organizado,24,83,-59,59
14053,licencias médicas,12,67,-55,55
4755,congreso nacional,30,84,-54,54


#### Compare ratio between groups
Which words are relatively more associated with one group more than the other?


In [20]:
def compare_groups_ratio(
    df,
    group_col,
    group_a,
    group_b,
    text_col="speech_clean",
    top_k=20,
    min_df=2,
    ngram_range=(1, 1)
):
    """
    Compare terms by relative association using smoothed count ratios.
    """
    
    # first get the raw count comparison
    result = compare_groups_totalwords(
        df=df,
        group_col=group_col,
        group_a=group_a,
        group_b=group_b,
        text_col=text_col,
        top_k=100000,   # keep many first, we trim later
        min_df=min_df,
        ngram_range=ngram_range
    ).copy()

    # add 1 so we do not divide by zero
    result[f"{group_a}_count_smooth"] = result[f"{group_a}_count"] + 1
    result[f"{group_b}_count_smooth"] = result[f"{group_b}_count"] + 1

    # ratio: how much more associated is the term with group A
    result["ratio_a_over_b"] = (
        result[f"{group_a}_count_smooth"] / result[f"{group_b}_count_smooth"]
    )

    # ratio: how much more associated is the term with group B
    result["ratio_b_over_a"] = (
        result[f"{group_b}_count_smooth"] / result[f"{group_a}_count_smooth"]
    )

    return result.sort_values("ratio_a_over_b", ascending=False).head(top_k)

In [21]:
compare_groups_ratio(analysis_df, group_col="sex_value", group_a="0", group_b="1", text_col="speech_clean", top_k=20, min_df=2, ngram_range=(2, 2))

,term,0_count,1_count,difference,abs_difference,0_count_smooth,1_count_smooth,ratio_a_over_b,ratio_b_over_a
15380,miguel ángel,25,0,25,25,26,1,26.0,0.038462
19822,proceso adopción,14,0,14,14,15,1,15.0,0.066667
16517,niños adolescentes,13,0,13,13,14,1,14.0,0.071429
9163,espinoza jorge,13,0,13,13,14,1,14.0,0.071429
14700,maría candelaria,12,0,12,12,13,1,13.0,0.076923
22823,santana castillo,11,0,11,11,12,1,12.0,0.083333
4564,concha smith,11,0,11,11,12,1,12.0,0.083333
2653,benavente vergara,11,0,11,11,12,1,12.0,0.083333
2650,becker alvear,11,0,11,11,12,1,12.0,0.083333
17037,olea joanna,11,0,11,11,12,1,12.0,0.083333


In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity

# def similarity_by_group(
#     df,
#     group_col,
#     text_col="speech_clean",
#     min_df=2,
#     ngram_range=(1, 1)
# ):
#     """
#     Compute cosine similarity between aggregated group texts.
#     """
#     grouped_text = df.groupby(group_col)[text_col].apply(lambda x: " ".join(x.dropna().astype(str)))

#     vec = TfidfVectorizer(
#         min_df=min_df,
#         ngram_range=ngram_range
#     )

#     X = vec.fit_transform(grouped_text)

#     sim = cosine_similarity(X)

#     return pd.DataFrame(sim, index=grouped_text.index, columns=grouped_text.index)

# similarity_by_group(analysis_df, group_col="region", text_col="speech_clean", min_df=2, ngram_range=(1, 1))

Full analysis

In [25]:
# full corpus
corpus_stats(analysis_df)

# top words
top_ngrams(analysis_df, text_col="speech_clean", top_k=15, min_df=2, ngram_range=(1, 1))

# bigrams
top_ngrams(analysis_df, text_col="speech_clean", top_k=15, min_df=2, ngram_range=(2, 2))

# tf-idf
compare_tfidf_groups(analysis_df, text_col="speech_clean", group_col="sex_value", group_a="1", group_b="0", top_k=20, min_df=2, ngram_range=(1,1))

# compare men vs women, if 1=men and 2=women
compare_groups_totalwords(analysis_df, group_col="sex_value", group_a="1", group_b="0", text_col="speech_clean", ngram_range=(1, 1), top_k=15)

,term,1_count,0_count,difference,abs_difference
7402,proyecto,1516,705,811,811
5383,ley,869,369,500,500
7133,presidente,628,275,353,353
6641,país,601,263,338,338
8463,si,522,187,335,335
4594,hoy,624,307,317,317
2844,dicho,554,244,310,310
4359,gobierno,404,98,306,306
8406,ser,488,195,293,293
9313,va,317,84,233,233
